# Parameter optimization of the denoising algorithm

## 1. Import libraries

In [15]:
import numpy as np
import pandas as pd

from src.evaluation.evaluation import read_results
from src.optimization.splits import load_split, split_results

## 2. Score definition

### 2.1. Per-artifact quality

$$
q_i = \frac{\text{clip}(\text{SNR}_i^{\text{out}}, F, C) - F}{C-F}
$$

where $F=-30$ dB and $C=0$ dB.

### 2.2. Global quality

$$
Q = \frac{1}{|\mathcal{S}| |\mathcal{A}| |\mathcal{C}|}\sum_{s\in\mathcal{S}}\sum_{a\in\mathcal{A}}\sum_{c\in\mathcal{C}} \bar{q}_{s,a,c}
$$

- $\mathcal{S}$, $\mathcal{A}$, $\mathcal{C}$: sets of SNR values, artifacts and clean signals.
- $\bar{q}_{s,a,c}$: mean quality of artifact $a$ in clean signal $c$ at SNR $s$ (mean over the positions of the artifact).

### 2.3. Number of false positives

$$
FP = \frac{1}{|\mathcal{C}|}\sum_{c\in\mathcal{C}}\bar{n}_c^{\text{false}}
$$

- $\bar{n}_c^{\text{false}}$: mean number of false positives of the mixtures of clean signal $c$.

### 2.4. Score

$$
S = Q - \lambda\cdot FP
$$

- $\lambda$: penalty of the false positives (see [3.2](#3.2.-Penalty-λ)).

### 2.5. Diagnostics

- Detection per SNR.
- SNR at 50 % detection.
- Conservation.
- False positives per label.
- Local RMS error of the detected artifacts.

In [1]:
F_DB = -30  # dB. Per-artifact quality: q = 0 at SNR_out <= F_DB ...
C_DB = 0    # dB. ... and q = 1 at SNR_out >= C_DB


def quality(snr_out_db, f=F_DB, c=C_DB):
    """
    Per-artifact quality: q = (clip(SNR_out, F, C) - F) / (C - F).

    Parameters
    ----------
    snr_out_db: array-like
        SNR in the artifact zone after the filter (dB)
    f, c: float
        Floor and ceiling of the SNR (dB): q = 0 at SNR_out <= f, q = 1 at SNR_out >= c

    Returns
    -------
    q: array-like
        Quality of each artifact, in [0, 1]
    """
    return (np.clip(snr_out_db, f, c) - f) / (c - f)


def snr_at_50(rate, levels):
    """
    SNR at which the detection rate crosses 0.5 (linear interpolation between levels), as in plot_detection of
    evaluation_denoise.ipynb. The detection decreases as the SNR increases: the last level with rate >= 0.5 is taken
    and interpolated up to the next one.
    +inf: >= 0.5 up to the last level (crossing above the range); -inf: < 0.5 in all of them (below the range).

    Parameters
    ----------
    rate: np.ndarray
        Detection rate at each SNR level
    levels: np.ndarray
        SNR levels (dB), in increasing order

    Returns
    -------
    snr50: float
        SNR at 50 % detection (dB)
    """
    above = rate >= 0.5
    if not above.any():
        return -np.inf
    i = np.where(above)[0].max()
    if i == len(rate) - 1:
        return np.inf
    r0, r1 = rate[i], rate[i + 1]
    return levels[i] + (0.5 - r0) * (levels[i + 1] - levels[i]) / (r1 - r0)


def score(results_df, lam, snr_list=None, f=F_DB, c=C_DB):
    """
    Score S = Q - λ·FP of the results of an experiment with one artifact per mixture (format of experiment_1.csv),
    and its diagnostics (see section 2):
    - Q: mean over the cells (SNR, artifact, clean signal) of the mean quality over the positions of the artifact.
      Only the existing cells are averaged (an artifact that does not fit in a clean signal has no cell)
    - FP: mean over the clean signals of the mean number of false positives of their mixtures (controls excluded)

    Parameters
    ----------
    results_df: pd.DataFrame
        Results (mixtures + controls), already restricted to the set to evaluate (e.g. tuning signals and artifacts)
    lam: float
        Penalty λ of the false positives
    snr_list: list or None
        SNR values (dB) of the set S. None -> all the SNR of the mixtures. FP uses the mixtures of the same SNR values
    f, c: float
        Floor and ceiling of the per-artifact quality (dB)

    Returns
    -------
    scores: dict
        S, Q, FP and lam
    diagnostics: dict
        - per_snr: pd.DataFrame, one row per SNR: number of mixtures, detection rate (mean over the cells, as Q),
          mean q (over the cells), mean q of the detected / undetected artifacts and dq = q_det - q_undet (cost of
          missing an artifact, see 3.2), median NMSE out of the artifact zone and median local RMS error (max, mean)
          after the filter of the detected artifacts
        - snr50: SNR at 50 % detection of all the artifacts (dB; ±inf: crossing out of the SNR range)
        - snr50_art: pd.Series, SNR at 50 % detection of each artifact
        - fp_label: pd.DataFrame, false positives per label of the clean signal: mean over its clean signals of the
          mean per mixture (as FP), mean per control and number of clean signals
        - nmse_control_db: median NMSE of the controls (whole signal)
    """
    df = results_df.assign(snr_db=pd.to_numeric(results_df["snr_db"], errors="coerce"))  # "Control" -> NaN
    is_control = df["art_id"].astype(str) == "Control"
    controls = df[is_control]
    mix = df[~is_control]
    if snr_list is not None:
        mix = mix[mix["snr_db"].isin(snr_list)]
    mix = mix.assign(q=quality(mix["snr_out_db"], f, c), detected=mix["detected"].astype(bool))

    # ============================================================================ score
    cells = ["snr_db", "art_id", "clean_id"]
    cell_q = mix.groupby(cells)["q"].mean()                   # q̄_{s,a,c}: mean over the positions
    Q = cell_q.mean()
    fp_clean = mix.groupby("clean_id")["n_false"].mean()      # n̄_c^false
    FP = fp_clean.mean()
    S = Q - lam * FP
    scores = dict(S=S, Q=Q, FP=FP, lam=lam)

    # ============================================================================ diagnostics
    # Detection, quality and conservation per SNR
    cell_det = mix.groupby(cells)["detected"].mean()
    det, undet = mix[mix["detected"]], mix[~mix["detected"]]
    per_snr = pd.DataFrame({
        "n_mixtures": mix.groupby("snr_db").size(),
        "detection": cell_det.groupby(level="snr_db").mean(),
        "q": cell_q.groupby(level="snr_db").mean(),
        "q_det": det.groupby("snr_db")["q"].mean(),
        "q_undet": undet.groupby("snr_db")["q"].mean(),
        "nmse_out_zone_db": mix.groupby("snr_db")["nmse_out_zone_db"].median(),
        "err_rms_max_loc_out": det.groupby("snr_db")["err_rms_max_loc_out"].median(),
        "err_rms_mean_loc_out": det.groupby("snr_db")["err_rms_mean_loc_out"].median(),
    })
    per_snr.insert(per_snr.columns.get_loc("q_undet") + 1, "dq", per_snr["q_det"] - per_snr["q_undet"])
    per_snr.index.name = "snr_db"

    # SNR at 50 % detection: all the artifacts and each artifact (detection rate = mean over the clean signals)
    levels = per_snr.index.to_numpy()
    snr50 = snr_at_50(per_snr["detection"].to_numpy(), levels)
    rate_art = cell_det.groupby(level=["art_id", "snr_db"]).mean().unstack("snr_db").reindex(columns=levels)
    snr50_art = rate_art.apply(lambda r: snr_at_50(r.to_numpy(), levels), axis=1).rename("snr50_db")

    # False positives per label of the clean signal
    label = df.groupby("clean_id")["label"].first()
    fp_label = pd.DataFrame({
        "fp_mixture": fp_clean.groupby(label).mean(),
        "fp_control": controls.groupby("label")["n_false"].mean(),
        "n_clean": fp_clean.groupby(label).size(),
    })
    fp_label.index.name = "label"

    diagnostics = dict(per_snr=per_snr, snr50=snr50, snr50_art=snr50_art, fp_label=fp_label,
                       nmse_control_db=controls["nmse_out_zone_db"].median())

    return scores, diagnostics

## 3. Phase 0: Preparation

- **Output:** tuning/test split, $\lambda$ and baseline.

### 3.1. Tuning/test split

75 % tuning / 25 % test (first of 4 folds as test, [get_splits.py](get_splits.py)), grouped by signal and artifact:

- **Signals:** stratified by label.
- **Artifacts:** stratified by duration.

| Set | Tuning | Test | Total |
|:----|:------:|:----:|:-----:|
| **Signals**   | 29 | 9 | 38 |
| **Artifacts** | 23 | 9 | 32 |

Signal × artifact combinations: **tuning** (29 × 23), **test** (9 × 9), **new signals** (9 × 23) and **new artifacts** (29 × 9).

In [16]:
clean_split, artifact_split = load_split()  # denoising/splits (generated by get_splits.py)

tune_clean = set(clean_split.loc[clean_split["set"] == "tune", "clean_id"])
test_clean = set(clean_split.loc[clean_split["set"] == "test", "clean_id"])
tune_art = set(artifact_split.loc[artifact_split["set"] == "tune", "art_id"])
test_art = set(artifact_split.loc[artifact_split["set"] == "test", "art_id"])

# Clean signals per label and set
signals = pd.crosstab(clean_split["label"], clean_split["set"], margins=True, margins_name="Total")
signals = signals[["tune", "test", "Total"]]
print("Clean signals per label")
display(signals)

# Artifacts per duration and set
artifacts = pd.crosstab(artifact_split["art_duration_ms"], artifact_split["set"], margins=True, margins_name="Total")
artifacts = artifacts[["tune", "test", "Total"]]
print("Artifacts per duration (ms)")
display(artifacts)

# Signal x artifact combinations of each set
combos = pd.DataFrame({
    "signals": [len(tune_clean), len(test_clean), len(test_clean), len(tune_clean)],
    "artifacts": [len(tune_art), len(test_art), len(tune_art), len(test_art)],
}, index=pd.Index(["tune", "test", "new_signals", "new_artifacts"], name="set"))
combos["combinations"] = combos["signals"] * combos["artifacts"]
print("Signal x artifact combinations")
display(combos)

# Ids of each set
print(f"Tuning clean signals: {sorted(tune_clean)}")
print(f"Test clean signals:   {sorted(test_clean)}")
print(f"Tuning artifacts:     {sorted(tune_art)}")
print(f"Test artifacts:       {sorted(test_art)}")

Clean signals per label


set,tune,test,Total
label,,,
Normal,7,3,10
Rhonchi,8,2,10
Stridor,6,2,8
Wheeze,8,2,10
Total,29,9,38


Artifacts per duration (ms)


set,tune,test,Total
art_duration_ms,,,
30.0,3,2,5
40.0,12,4,16
50.0,6,1,7
60.0,1,0,1
70.0,0,1,1
80.0,1,0,1
90.0,0,1,1
Total,23,9,32


Signal x artifact combinations


,signals,artifacts,combinations
set,,,
tune,29,23,667
test,9,9,81
new_signals,9,23,207
new_artifacts,29,9,261


Tuning clean signals: [0, 1, 3, 4, 5, 6, 8, 10, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 31, 32, 35, 36, 37]
Test clean signals:   [2, 7, 9, 11, 15, 22, 26, 33, 34]
Tuning artifacts:     [0, 1, 2, 3, 5, 7, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28]
Test artifacts:       [4, 6, 8, 13, 20, 26, 29, 30, 31]


In [23]:
results_df = read_results("experiment_1.csv")

# Split into the four sets of section 3.1: each one keeps the controls of its clean signals
sets = split_results(results_df, clean_split, artifact_split)
is_control = results_df["art_id"].astype(str) == "Control"

# Summary of each set
summary = {}
for name, df in sets.items():
    ctrl = df["art_id"].astype(str) == "Control"
    summary[name] = {
        "signals": df["clean_id"].nunique(),
        "artifacts": df.loc[~ctrl, "art_id"].nunique(),
        "combinations": df[~ctrl].groupby(["clean_id", "art_id"]).ngroups,
        "mixtures": (~ctrl).sum(),
        "controls": ctrl.sum(),
    }
summary = pd.DataFrame(summary).T
summary.index.name = "set"
display(summary)

tune = sets["tune"]
test = sets["test"]
new_signals = sets["new_signals"]
new_artifacts = sets["new_artifacts"]

,signals,artifacts,combinations,mixtures,controls
set,,,,,
tune,29,23,667,26664,29
test,9,9,81,3240,9
new_signals,9,23,207,8280,9
new_artifacts,29,9,261,10432,29


In [24]:
tune

,id,clean_id,clean_file,label,art_id,rep,snr_db,k0,k1,t0_s,...,attenuated,nmse_out_zone_db,err_rms_max_loc_mix,err_rms_max_loc_out,err_rms_mean_loc_mix,err_rms_mean_loc_out,err_rms_max_glob_mix,err_rms_max_glob_out,err_rms_mean_glob_mix,err_rms_mean_glob_out
0,0,0,19,Normal,0,0.0,-30,2224.0,2344.0,0.556,...,True,-48.357410,2276.685910,-18.708341,1362.163855,-11.608118,1469.912947,-5.463990e-05,63.351784,-5.375969e-01
1,1,0,19,Normal,0,0.0,-20,2224.0,2344.0,0.556,...,True,-49.934976,649.297471,-13.237228,388.410368,-4.462894,394.946259,-1.727932e-05,18.064209,-2.142408e-01
2,2,0,19,Normal,0,0.0,-10,2224.0,2344.0,0.556,...,False,-50.061333,143.788973,143.788973,92.022027,92.022027,61.034095,6.103410e+01,4.279768,4.273706e+00
3,3,0,19,Normal,0,0.0,-5,2224.0,2344.0,0.556,...,False,-50.076448,48.527876,48.527876,38.867207,38.867207,-0.000003,-3.072792e-06,1.807643,1.801581e+00
4,4,0,19,Normal,0,0.0,0,2224.0,2344.0,0.556,...,False,-50.081239,8.329665,8.329665,14.160422,14.160422,-0.000002,-1.727960e-06,0.658579,6.525167e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48647,48671,31,9423,Stridor,Control,NaN,Control,NaN,NaN,NaN,...,NaN,-59.446377,NaN,NaN,NaN,NaN,0.000000,1.807245e-03,0.000000,-4.734672e-04
48648,48672,32,9735,Stridor,Control,NaN,Control,NaN,NaN,NaN,...,NaN,-19.918233,NaN,NaN,NaN,NaN,0.000000,-1.209588e-14,0.000000,-5.146678e+00
48651,48675,35,17812,Stridor,Control,NaN,Control,NaN,NaN,NaN,...,NaN,-74.983649,NaN,NaN,NaN,NaN,0.000000,-1.308418e-14,0.000000,4.221891e-05
48652,48676,36,23218,Stridor,Control,NaN,Control,NaN,NaN,NaN,...,NaN,-312.018659,NaN,NaN,NaN,NaN,0.000000,-1.293224e-14,0.000000,-1.884035e-13


### 3.2. Penalty λ

One false positive is worth missing an artifact right at the masking limit $s^*$.

**Masking SNR $s^*$:**

- **Masking:** the artifact raises the global RMS peak (moving RMS, 30 ms) by more than 10 % (`err_rms_max_glob_mix` > 10 %).
- $s^*$ = SNR at which 50 % of the mixtures are masked (linear interpolation between the SNR of `SNR_LIST`).

**Cost of missing an artifact at $s^*$**, computed in two ways:

1. **Approximation (ideal elimination, $q^{\text{det}} = 1$):**

$$
\lambda \approx -\frac{s^*}{30}
$$

2. **Measured** ($q^{\text{det}} < 1$):

$$
\Delta q_s = \hat{q}^{\text{det}}_s - \hat{q}^{\text{undet}}_s, \qquad \lambda = \Delta q_{s^*}
$$

where $\hat{q}^{\text{det}}_s$ / $\hat{q}^{\text{undet}}_s$ are the mean $q$ of the detected / undetected artifacts at SNR $s$ ($\Delta q_{s^*}$ interpolated linearly between the SNR of `SNR_LIST`).

- Computed with the current results (`experiment_1.csv`), on the tuning set only.

### 3.3. Baseline

Score, Q and FP of the current version on the tuning set, with the $\lambda$ of [3.2](#3.2.-Penalty-λ).

## 4. Phase 1: Candidate generation

- **Goal:** almost every artifact that matters should generate some candidate, without an excessive load on the clean signals and without overflowing the artifact.
- **Optimization of:** `WAVELET`, `K`, `ENV_SMOOTH`, `MASK_DILATION`, `REF_INNER`, `REF_OUTER`, `MAX_EVENT`.
- **Output:** C1.

### 4.1. Metrics

Measured apart and with priority (a missed candidate is lost for good; a wider span only degrades its features):

1. **Candidate recall:** proportion of artifacts with SNR ≤ 0 dB that have some candidate in their zone $Z_i$ (artifact ± 25 ms).
2. **Load:** candidates per control signal.
3. **Overflow:** median of $w_i$ over the artifacts that have some candidate:

$$
w_i = \frac{|Z_i|}{|Z_i \cup \text{spans}_i|} = \frac{|Z_i|}{|Z_i| + \text{overflow}_i}
$$

where $\text{spans}_i$ is the union of the candidate spans that touch $Z_i$.

- **Diagnostics:** median overflow (ms) and weighted candidate recall ($\text{Recall}_w = \frac{1}{N}\sum_i w_i$, with $w_i = 0$ if there is no candidate).

### 4.2. Screening

Each parameter is varied alone. If the metrics do not change, it is fixed.

### 4.3. Search

Two Bayesian optimizations, one after the other:

- **(a) Maximum recall:** maximize the recall subject to load < 2 × baseline → recall$_{\max}$.
- **(b) Minimum load:** minimize the load subject to recall ≥ recall$_{\max}$ − 0.005 and median $w_i$ ≥ median $w_i$ of the baseline.

The recall, load and overflow of every evaluated configuration (of both searches) are saved.

### 4.4. Choice

Among all the evaluated configurations (a and b) that meet the two constraints of (b), the one with the minimum load → **C1**.

- **Check:** load–overflow Pareto front of the configurations with recall ≥ recall$_{\max}$ − 0.005 (e.g. whether a little more load would give much less overflow).

### 4.5. Full run

Computation of the estimates needed for the Score in [Phase 3](#6.-Phase-3:-Rule-selection).

## 5. Phase 2: Feature definition

- **Goal:** measure each feature so that it separates the artifacts from the rest as well as possible. The candidates are those of C1 and do not change.
- **Metric:** standardized partial ROC-AUC with a false positive rate ≤ 0.1, within the cases where each feature is used.
- **Output:** candidate table.

### 5.1. Width

- **Features:** `width_med`, `width_max`.
- **Parameters:** `BG_FACTOR`, `WIDTH_FLOOR_DB`, `EXTEND`.

### 5.2. Isolation

- **Feature:** `isolation`.
- **Parameter:** `ISO_WIN`.

### 5.3. Impulsiveness

- **Feature:** `z_max`.
- **Parameters:** `Z_WIN`, `AR_ORDER`.

### 5.4. Choice

Combination with the best mean partial AUC of each group (AUC vs value of each parameter).

### 5.5. Candidate table

Features of every candidate with the chosen parameters.

## 6. Phase 3: Rule selection

- **Goal:** define the rule matrix: 4 cases × 8 features (see [6️⃣ Click rules](README.md#6️⃣-click-rules)).
- **Metric:** $\hat{S}$ (estimated score).
- **Output:** final rule structure and thresholds.

### 6.1. Descriptive analysis

- Distributions of the features per class.
- Pairwise scatter plots per class.
- Correlation matrix.

### 6.2. Univariate analysis

For each case:

1. **Filter** with the main condition of the case.
2. **Minimum condition:** $SE \leq 0.03$ (bootstrap grouped by signal and by artifact); otherwise, relax the filter one percentile and repeat (adaptive margin at the relaxation limit: $|AUC - 0.5| \geq \max(0.1,\ 3 \cdot SE)$).
3. **Prefilter:** AUC of each of the other 7 features → condition ≥ / ≤ / not added ($|AUC - 0.5| \leq 0.1$).

**Output:** AUC matrix (case × feature).

### 6.3. Provisional optimization

$$
\hat{S} = \frac{1}{|\mathcal{S}|}\sum_{s\in\mathcal{S}}\left[\left(1-\widehat{\text{Det}}_s\right)\cdot\hat{q}_s^{\text{undet}} + \hat{q}_s^{\text{det}}\cdot\widehat{\text{Det}}_s\right] - \lambda \cdot \widehat{FP}
$$

1. **Search:** Bayesian optimization over the conditions that passed the prefilter, maximizing $\hat{S}$.
2. **Cross-validation:** 5 folds → 5 sets of thresholds + 5 values of $\hat{S}$.

### 6.4. Importance of each condition

| Measure | Definition |
|:--------|:-----------|
| **Unique rejections** | Number of candidates that meet every condition except this one and that this condition rejects |
| **Ablation** | $\Delta\hat{S}$ when it is disabled |
| **Stability** | Variability of the threshold across folds |
| **Unique detections per case** | Number of artifacts detected by that case and by no other |

**Output:** importance table (initial).

### 6.5. Backward pruning

1. **Pruning:** remove the condition with the highest $\Delta\hat{S}$.
2. **Optimization:** re-optimize the remaining thresholds.
3. **Repeat** until reaching a rule with a single condition.
4. **1-SE rule:** choose the simplest rule set whose $\hat{S}$ is within 1 SE of the best.
5. **Final thresholds:** optimize the thresholds of the remaining conditions on the whole tuning table.

**Outputs:** pruning curve, importance table (final), final rule structure and thresholds.

## 7. Phase 4: Attenuation

- **Optimization of:** `G_FAC`, `GAIN_SMOOTH`, `LOCAL_BG`.
- **Metric:** Score.
- **Output:** the optimized version.

## 8. Phase 5: Validation

- **Metrics:** S, Q, FP.
- **Output:** comparison and generalization.

### 8.1. Sensitivity to λ

Rule selection as a function of the value of $\lambda$.

### 8.2. Comparison

- S (Q, FP) for different values of $\lambda$.
- Wilcoxon signed-rank test, 95 % bootstrap CI of ΔS and number of signals that improve.

### 8.3. Generalization

Optimism = ΔS of the tuning set − ΔS of the test sets (test, new signals and new artifacts).